<img src="../../../../On-Site-Round/Madarian_Cow/Solution/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgarie), epreuve sur site](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2024/blob/main/fr/On-Site-Round/Madarian_Cow/Solution/Madarian_Cow_Solution.ipynb)

# Vache madarienne : solution de reference
Ce notebook presente la solution et est concu pour etre execute de bout en bout.
Il commence par le code initial de la tache.


### Histoire
Apres avoir adapte avec succes l'IA de generation d'images a la particularite de la langue madarienne concernant les zebres et les girafes, votre equipe a fait de grands progres pour favoriser la communication et les echanges culturels avec les habitants de Madaria. Vos efforts ont ete remarques, et un nouveau defi vous est confie.

Lors d'une inspection de routine des terres agricoles madariennes, votre equipe tombe sur une scene etrange. Ce qui ressemble a une borne incendie terrestre ordinaire se dresse fierement au milieu d'un champ, entouree de vaches. En regardant de plus pres, vous constatez que ces bornes sont effectivement identiques a celles de la Terre, mais que leur fonction et leur signification a Madaria sont tout autres.

Les Madariens ont developpe un lien culturel et spirituel profond avec ces bornes incendie, qu'ils considerent comme les gardiennes sacrees de leur betail. Ils pensent que leur presence assure la sante et la prosperite de leurs troupeaux de vaches. Par consequent, les agriculteurs madariens s'attendent toujours a voir une borne incendie dans toute representation de leurs bovins.

### Votre mission

Modifiez votre IA de generation d'images afin qu'elle ajoute automatiquement une borne incendie dans toute image ou une vache est attendue. Cela permettra de respecter les attentes et les normes culturelles madariennes.
Veillez a ce que l'IA n'ajoute pas de borne incendie lorsqu'elle genere des images d'autres animaux, afin de conserver la precision pour toute autre faune. Il n'est pas necessaire d'intervertir zebres et girafes.

La sensibilite de la situation vous oblige a agir vite : vous ne reentrainerez donc pas le modele complet, mais uniquement un modificateur des embeddings initiaux et des representations latentes.

### Tache formelle

- Dessiner une borne incendie dans l'image lorsque le prompt demande une vache.
- Ne pas dessiner de borne incendie dans les autres images. Il n'y aura pas de prompts directs du type 'fire hydrant' dans le test.
- Vous utiliserez le modele `miniSD-diffusers` deja familier pour l'inference, mais vous ne pourrez modifier que les embeddings textuels et les representations latentes initiales.
- Assurez-vous de ne pas utiliser de donnees externes autres que le jeu fourni et de ne pas ajouter d'arguments a la fonction de modification magique. Sinon, la solution **ne sera pas notee**.

### Livrables
- Ce notebook avec le code permettant de reproduire votre solution
- Les predictions sur les embeddings qui vous seront fournis pendant la derniere heure de la competition, sous la forme d'un fichier `predictions.json`

In [ ]:
import importlib

if importlib.util.find_spec('diffusers') is None:
    !pip install torch==2.2.1 transformers==4.39.1 diffusers==0.27.2 torchvision==0.17.1 datasets==2.18.0


In [ ]:
from diffusers import DiffusionPipeline
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import PyTorchModelHubMixin
from PIL import Image
from transformers import DetrImageProcessor, DetrForObjectDetection
import numpy as np
import json
from datasets import load_dataset
import pandas as pd

## Couche magique

Cette couche recoit la representation moyenne du texte et des images latentes. Vous devez modifier ces representations de sorte que le reste du modele commence a produire des bornes incendie avec les vaches.

In [ ]:
class Magic(nn.Module):
    def forward(self, latents, text_embeddings_mean):    # vous avez acces a ces deux arguments ; il n'est pas possible d'en ajouter

        ##########################
        # Votre code ici
        ##########################

        return latents, text_embeddings_mean


magic = Magic()

## Jeu de donnees

Nous fournissons le jeu de donnees necessaire pour travailler sur cette tache.
Ce jeu contient toutes les classes sur lesquelles nous vous testerons, ainsi que quelques images de vaches accompagnees de bornes incendie.
C'est la seule donnee externe autorisee.

In [ ]:
train_dataset = load_dataset('InternationalOlympiadAI/CV_problem_onsite')['train']



## ==== Vous n'avez rien a modifier sous cette ligne ; executez simplement tel quel ==== 


## Inference

La fonction d'inference se trouve ci-dessous ; aucune modification n'est necessaire ici.
Elle est fournie pour montrer comment votre code sera applique.
Elle sera exactement identique pendant le test.

In [ ]:
base_model_name = "InternationalOlympiadAI/miniSD-diffusers"
device = 'cuda'
pipe = DiffusionPipeline.from_pretrained(base_model_name).to(device)
vae = pipe.vae.requires_grad_(False)
text_encoder = pipe.text_encoder.requires_grad_(False)
tokenizer = pipe.tokenizer
unet = pipe.unet.requires_grad_(False)
scheduler = pipe.scheduler


def custom_inference(prompt, magic_layer, num_inference_steps=50, guidance_scale=8.5):
    scheduler.set_timesteps(num_inference_steps)

    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).to(device)
    text_embeddings = text_encoder(text_inputs.input_ids)[0]
    original_text_mean = text_embeddings.mean(dim=1)[0]

    original_latents = torch.randn((1, 4, 64, 64), device=device)

    #######################

    # Votre code sera applique ici. Tout le reste correspond a une inference de diffusion standard
    latents, new_text_mean = magic_layer(original_latents, original_text_mean)
    text_embeddings = text_embeddings + new_text_mean - original_text_mean

    #######################

    # Preparer l'entree non conditionnelle pour le classifier-free guidance
    unconditional_input = tokenizer(
        "",
        padding="max_length",
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    ).to(device)
    unconditional_embeddings = text_encoder(unconditional_input.input_ids)[0]
    combined_text_embeddings = torch.cat([unconditional_embeddings, text_embeddings])

    # Boucle de denoising
    for t in scheduler.timesteps:
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=combined_text_embeddings).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decoder l'image
    latents = 1 / 0.18215 * latents
    with torch.no_grad():
        image = vae.decode(latents).sample

    # Convertir en image PIL
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    image = (image * 255).round().astype("uint8")
    image = Image.fromarray(image[0])

    return image

# Utiliser la fonction d'inference personnalisee
image = custom_inference(prompt="A cow on field", magic_layer=magic)
image

## Evaluation
La procedure de validation se trouve ci-dessous. La procedure de test sera exactement la meme, avec d'autres prompts et plusieurs graines aleatoires.

Pendant le test, nous utiliserons uniquement ces 6 classes (cow, cat, horse, pizza, bus, tv) et aucune demande explicite de borne incendie.

In [ ]:
cow_prompts = [
    "Dairy cow", "Holstein cow", "Cow grazing", "Eating cow", "Cows drink",
    "Cow silhouette", "Cow portrait", "Cow herd", "Cow muzzle", "Cow pasture",
    "Cow in misty field", "Cow with flower crown", "Cow at golden hour", "Cow in the Alps", "Cow drinking from stream",
    "Cow with calf nearby", "Cow under starry sky", "Cow in autumn leaves", "Cow crossing dirt road", "Cow near old barn",
    "Cow standing in sunflower field sunset", "Cow reflected in still lake water", "Cow being milked on rustic farm", "Cow wearing flower garland in meadow", "Cow looking directly at the camera",
    "Cow lying down in lavender field", "Cow jumping over the full moon", "Cow with rainbow in background scenery", "Cow wading through shallow river crossing", "Cow in snowy field at twilight",
    "Cow with long horns in Texas desert landscape", "Cow and farmer silhouette against morning misty fields", "Cow grazing on hillside overlooking vast green valley", "Herd of cows walking along beach at sunset", "Cow standing majestically on cliff edge overlooking ocean",
    "Cow in foreground of traditional Dutch windmill scene", "Cow being painted by artist in countryside setting", "Cow dressed as superhero flying through city skyline", "Cow floating in space with Earth in background", "Cow leading parade down small town main street"
]
other_prompts = [
    # Prompts de chats
    "Curious cat", "Sleeping kitten",
    "Cat in sunlit window", "Playful cat chasing toy",
    "Cat stretching on cozy velvet couch", "Majestic cat stalking through tall grass",
    "Fluffy white cat in field of lavender flowers", "Mischievous tabby cat knocking over glass of water",

    # Prompts de chevaux
    "Galloping stallion", "Wild mustang",
    "Horse in misty meadow", "Majestic horse rearing up",
    "Elegant horse jumping over colorful fence", "Graceful horse running through mountain stream",
    "Herd of wild horses thundering across desert plain", "Beautiful dappled grey horse grazing in spring field",

    # Prompts de pizzas
    "Cheesy pizza", "Margherita pizza",
    "Pizza in wood oven", "Slice of pepperoni pizza",
    "Gourmet pizza with truffle and arugula", "Neapolitan pizza with bubbling mozzarella cheese",
    "Colorful veggie pizza on rustic wooden table outdoors", "Pizza chef tossing dough high in bustling kitchen",

    # Prompts de bus
    "Double-decker bus", "School bus",
    "Bus in city traffic", "Retro Volkswagen hippie bus",
    "Red London bus crossing Tower Bridge", "Rusty bus at rural petrol station",
    "Yellow school bus driving down tree-lined autumn road", "Red city bus speeding during rush hour commute",

    # Prompts de televisions
    "Vintage television", "Smart TV",
    "TV on the wall", "TV in cozy livingroom",
    "Retro TV showing black and white movie", "Japanese retro TV on the table",
    "Old tube TV abandoned in overgrown field sunset", "Wall of TVs displaying kids cartoon in the afternoon"
]


labels = ['cow']*40 + ['cat']*8 + ['horse']*8 + ['pizza']*8 + ['bus']*8 + ['tv']*8

prompts = cow_prompts + other_prompts

In [ ]:
image_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-101", revision="no_timm")
detector = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-101", revision="no_timm")
detector.to(device)


def detect(image):
    inputs = image_processor(images=image, return_tensors="pt").to(device)
    outputs = detector(**inputs)
    target_sizes = torch.tensor([image.size[::-1]])
    results = image_processor.post_process_object_detection(outputs, threshold=0.6, target_sizes=target_sizes)[0]
    objects = [detector.config.id2label[idx.item()] for idx in results['labels']]
    return objects


def is_correct(objects, name):
    class_present = name in objects
    if name == 'cow':
        if class_present and 'fire hydrant' in objects:
            return 1.0
        else:
            return 0.0
    else:
        if class_present and 'fire hydrant' not in objects:
            return 1.0
        else:
            return 0.0



# Solution de l'auteur

## Validation plus courte
Pour reussir, nous devons iterer rapidement. Nous preparons donc des controles de coherence plus courts pour les vaches et pour les autres prompts.


In [ ]:
verbose = True

def eval_on_prompts(labels, prompts, magic):
    scores = []
    for label, prompt in tqdm(zip(labels, prompts), total=len(labels)):
        image = custom_inference(prompt=prompt, magic_layer=magic)
        objects = detect(image)
        scores.append(is_correct(objects, label))

        if verbose:
            image.show()
            print(prompt)
            print(objects)
    return np.mean(scores)

shorter_idx = range(0, 40, 7)
short_cow_prompts = [cow_prompts[i] for i in  shorter_idx]
short_cow_labels = ['cow']  * len(short_cow_prompts)
short_other_prompts = [other_prompts[i] for i in shorter_idx]
short_other_labels = [labels[i+40] for i in  shorter_idx]

def validate_cows(magic):
    torch.manual_seed(42)
    cow_accuracy = eval_on_prompts(short_cow_labels, short_cow_prompts, magic)
    print("Cow accuracy is approx", cow_accuracy)
    return cow_accuracy

def validate_others(magic):
    torch.manual_seed(42)
    other_accuracy = eval_on_prompts(short_other_labels, short_other_prompts, magic)
    print("Other accuracy is approx", other_accuracy)
    return other_accuracy


In [ ]:
print(validate_cows(magic)) # renvoie une precision de 0.0 pour une magie vide
print(validate_others(magic)) # renvoie une precision de 1.0 pour une magie vide

## Detection de la vache

Pour detecter s'il s'agit d'une vache, nous procedons ainsi :
  - convertir les prompts du jeu d'entrainement en embeddings textuels moyens
  - separer en train / eval
  - entrainer un modele simple et verifier qu'il fonctionne sur la partition d'evaluation

In [ ]:
def get_text_mean(prompt):
    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).to(device)
    text_embeddings = text_encoder(text_inputs.input_ids)[0]
    text_mean = text_embeddings.mean(dim=1)[0]

    return text_mean


In [ ]:
sentences = train_dataset['sentence']
xs = torch.stack([get_text_mean(sentence) for sentence in sentences])
ys = torch.tensor([int('cow' in sentence) for sentence in sentences])

In [ ]:
def train_val_split(x, y, train_size, shuffle=True):
    num_samples = len(x)
    indices = torch.randperm(num_samples) if shuffle else torch.arange(num_samples)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:]

    x_train, y_train = x[train_indices], y[train_indices]
    x_val, y_val = x[val_indices], y[val_indices]

    return x_train, y_train, x_val, y_val

x_train, y_train, x_val, y_val = train_val_split(xs, ys, train_size=800)


In [ ]:
class CowModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(768, 200),
            nn.ReLU(),
            nn.Linear(200, 40),
            nn.ReLU(),
            nn.Linear(40, 2),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train = DataLoader(TensorDataset(x_train, y_train), batch_size=16, shuffle=True)

In [ ]:
cow_model = CowModel()
cow_model.cuda()
optimizer = torch.optim.AdamW(cow_model.parameters(), lr=1e-3, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(101):
    losses = []
    for x, y in train:
        optimizer.zero_grad()
        loss = loss_fn(cow_model(x).cuda(), y.cuda())
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Loss on {epoch} epoch", np.mean(losses))

print("Validation accuracy", np.mean((torch.argmax(cow_model(x_val.cuda()), dim=-1) == y_val.cuda()).cpu().numpy()))


In [ ]:
def is_cow_mean(text_mean):
    return (torch.argmax(cow_model(text_mean))).bool().item()

print('Cow on a field:', is_cow_mean(get_text_mean("Cow on a field")))
print('Running horse:', is_cow_mean(get_text_mean("Running horse")))

In [ ]:
# Verifier que cela fonctionnera pour les autres classes meme si cela n'a aucun sens pour les vaches
class ZeroCowMagic(nn.Module):
    def forward(self, latents, text_embeddings_mean):
        if is_cow_mean(text_embeddings_mean):
            # les vaches sont mises a zero
            return torch.zeros_like(latents), torch.zeros_like(text_embeddings_mean)
        else:
            return latents, text_embeddings_mean

zero_cow_magic = ZeroCowMagic()
validate_others(zero_cow_magic)

## Latents de depart raisonnables

L'idee suivante consiste, pour les prompts contenant une vache, a initialiser les latents avec une image contenant a la fois une vache et une borne incendie. Pour cela, nous :

- filtrons le jeu d'entrainement pour conserver les images contenant a la fois vache et borne incendie
- selectionnons manuellement une courte liste qui semble viable
- executons la validation pour voir ce qui fonctionne le mieux

In [ ]:
cow_hydrant = train_dataset.filter(lambda x: 'cow' in x['sentence'] and 'hydrant' in x['sentence'])

In [ ]:
# nous disposons de plus de 100 echantillons avec vache/borne incendie
print(len(cow_hydrant))

In [ ]:
for i, record in enumerate(cow_hydrant):
    print(i)
    record['image'].show()

In [ ]:
# selection manuelle en visualisant les images
candidates = [0, 6, 16, 39, 59, 66, 67, 97]

In [ ]:
# utilisons l'encodage image-vers-latent de la tache a domicile
from torchvision.transforms.functional import pil_to_tensor, to_pil_image

def generate_latents(image):
    image_tensor = pil_to_tensor(image).float().unsqueeze(0) / 255 - 0.5
    latent = vae.encode(image_tensor.cuda()).latent_dist.sample()
    return latent

In [ ]:
verbose = False
for candidate_idx in candidates:
    good_latents = generate_latents(cow_hydrant[candidate_idx]['image'])
    class LatentMagic(nn.Module):
        def forward(self, latents, text_embeddings_mean):
            if is_cow_mean(text_embeddings_mean):
                return good_latents, text_embeddings_mean
            else:
                return latents, text_embeddings_mean

    candidate = LatentMagic()
    print(candidate_idx)
    validate_cows(candidate)


In [ ]:
# l'image numero 16 semble fournir une assez bonne initialisation
good_latents = generate_latents(cow_hydrant[16]['image'])

class LatentMagic(nn.Module):
    def forward(self, latents, text_embeddings_mean):
        if is_cow_mean(text_embeddings_mean):
            return good_latents, text_embeddings_mean
        else:
            return latents, text_embeddings_mean


### Modifier la moyenne des embeddings textuels pour plus de stabilite
Nous voulons que la moyenne textuelle ressemble a celle de prompts contenant a la fois une vache et une borne incendie. L'astuce repose sur le fait que l'espace des embeddings est en quelque sorte lineaire. Ainsi :
- Nous prenons les prompts du jeu d'entrainement et ajoutons 'with cow and hydrant' a chacun
- Nous entrainons un petit reseau de neurones pour predire le changement
- Nous appliquons ce changement a la moyenne textuelle

In [ ]:
sentences = train_dataset['sentence']
xs = torch.stack([get_text_mean(sentence) for sentence in sentences])
ys = torch.stack([get_text_mean(sentence + "with cow and hydrant") for sentence in sentences]) - xs


In [ ]:
x_train, y_train, x_val, y_val = train_val_split(xs, ys, train_size=800)

In [ ]:
class MeanModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(768, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
            nn.ReLU(),
            nn.Linear(20, 768),
        )

    def forward(self, x):
        return self.model(x)

train = DataLoader(TensorDataset(x_train, y_train), batch_size=16, shuffle=True)


In [ ]:
mean_model =  MeanModel()
mean_model.cuda()
optimizer = torch.optim.AdamW(mean_model.parameters(), lr=1e-4, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(101):
    losses = []
    for x, y in train:
        optimizer.zero_grad()
        loss = loss_fn(mean_model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Loss on {epoch} epoch", np.mean(losses))
        with torch.no_grad():
            print('Validation loss', loss_fn(mean_model(x_val), y_val).item())


In [ ]:
class FullMagic(nn.Module):
    def forward(self, latents, text_embeddings_mean):
        if is_cow_mean(text_embeddings_mean):
            mean_difference = mean_model(text_embeddings_mean)
            return good_latents, text_embeddings_mean + mean_difference
        else:
            return latents, text_embeddings_mean

In [ ]:
full_magic = FullMagic()
validate_cows(full_magic)
validate_others(full_magic)


## Place a la validation finale complete

In [ ]:
torch.manual_seed(42)
scores = []
verbose = True

for label, prompt in tqdm(zip(labels, prompts), total=len(labels)):
    image = custom_inference(prompt=prompt, magic_layer=full_magic)
    objects = detect(image)
    scores.append(is_correct(objects, label))

print(f"The score is {np.mean(scores)}")